## Notebook 6: When Accurate Models Produce Wrong Science

 **Audience:** First-year PhD students in Astrophysics  
 **Theme:** Why models that look good can still produce bad science

###### Astrophysics context

In modern surveys such as SDSS, DES, Euclid, Rubin/LSST, and Roman, machine learning is routinely used to estimate physical quantities:

- photometric redshift
- stellar parameters
- morphology
- transient class labels

However, many failures in scientific ML do **not** happen because the algorithm is weak.

They happen because:

- training labels are biased
- data distributions shift
- uncertainty is ignored
- hidden variables are missing
- models are used outside their domain of validity

###### Scientific goal

Understand:

- selection bias
- distribution shift
- interpolation vs extrapolation
- overconfidence
- why uncertainty matters for science

In astronomy, data limitations often dominate model choice.

> A good model trained on bad data is still a bad model.

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from matplotlib.colors import LinearSegmentedColormap
from ngboost import NGBRegressor
from ngboost.distns import Normal
from sklearn.tree import DecisionTreeRegressor

## Set Random Seed and Plotting Style

This step initializes reproducibility and configures the default visualization settings.


The plotting style is also set to a predefined Matplotlib theme that improves readability and accessibility. Default figure size and font size are adjusted to make all plots clearer and more consistent throughout the notebook.

These settings ensure that results are both reproducible and visually consistent across all visualizations.

In [ ]:
SEED = 42
np.random.seed(SEED)
plt.style.use("tableau-colorblind10")
plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["font.size"] = 12

## Scientific Context: Photometric Redshifts as an Inverse Problem

Modern surveys (LSST, Euclid, Roman) infer physical quantities—redshift ($z$), stellar age ($T$), or dust ($A_v$)—from broad-band photometry. 

This is a classic **inverse problem** governed by the transition from high-dimensional spectra to low-dimensional observables.

###### The Mapping: Forward vs. Inverse

* **Forward Problem: $z \rightarrow x_{\mathrm{photo}}$**
  - Given a galaxy's physical state, what colors will the telescope see?<br>This mapping is determined by galaxy SEDs, cosmic expansion, dust attenuation, and filter response functions.
* **Inverse Problem: $x_{\mathrm{photo}} \rightarrow p(z \mid x_{\mathrm{photo}})$**
  - Given a few noisy magnitudes (e.g., $u,g,r,i$), what was the physical state?<br>This is fundamentally probabilistic because the process is **lossy**.

###### Information Loss and Degeneracy

By squashing a spectrum into a few broad filters, we project a high-dimensional physical space into a low-dimensional observational space.

This creates **degeneracy**, where different physical states—such as a dusty low-$z$ galaxy and a young high-$z$ galaxy—look identical to the telescope.

###### Unified Formalism

Conceptually, these experiments relate to the conditional inference problem:

$$p(z \mid x_{\mathrm{photo}}, \mathcal{D}_{\mathrm{train}})$$

Where the mapping is defined by:

1. **Observation Model**: 

$$x_{\mathrm{photo}} = g(z, \epsilon)$$ 
   - where $g$ is the lossy telescope pipeline and $\epsilon$ is noise.
   - Because $g$ is not injective, the inverse problem is fundamentally probabilistic rather than uniquely invertible.
2. **Training Support**: 

$$\mathcal{D}_{\mathrm{train}} \sim p_{\mathrm{train}}(x,z)$$

   - In astronomy, spectroscopic labels are often magnitude-limited and biased toward bright, low-redshift objects.

> We aren't just fitting a line; we are reconstructing the state of the universe from blurry, noisy pixels.

If $\mathcal{D}_{\mathrm{train}}$ is incomplete or $x_{\mathrm{photo}}$ are degenerate, the models cannot reliably infer unconstrained structure without additional assumptions.

## Data-Generating Process and Stochastic Ambiguity

We simulate a simplified photometric-redshift relation:

* **True Redshift:** $z \sim U(0,2)$
* **Observable Feature:** $\text{color} = 1.5 \log(1+z) + \epsilon$

The Gaussian noise $\epsilon$ represents measurement uncertainty and unmodeled physics. 

This process renders the mapping **nonlinear**, **stochastic**, and **imperfectly invertible**.

Notice that this toy model captures support mismatch and uncertainty failure, but not the full multimodal degeneracies of realistic photometric-redshift inference.

### Stochastic vs. Physical Ambiguity

In a real-world scenario, ambiguity is often driven by **physical degeneracy** (e.g., dust–redshift or age–metallicity degeneracies), where identical colors correspond to different physical states even without noise.

Our toy model is simpler: *the underlying relation is monotonic, but observational noise broadens the mapping*.

> Even without complex physical degeneracies, observational noise and incomplete training support are sufficient to produce unreliable inference under distribution shift.

Similar observed colors will correspond to a range of plausible redshifts, turning a deterministic physical value into a probabilistic guess.

In [ ]:
N = 9000
N_test = 9000

z = np.random.uniform(0.0, 2.0, N)
z_test = np.random.uniform(0.0, 2.0, N_test)

color = 1.5 * np.log1p(z) + np.random.normal(0, 0.15, N)
color_test = 1.5 * np.log1p(z_test) + np.random.normal(0, 0.15, N_test)

df = pd.DataFrame({ "color": color, "z": z })
df.head()

In [ ]:
plt.scatter(df["z"], df["color"], s=8, alpha=0.25)
plt.xlabel("True redshift")
plt.ylabel("Observed colour")
plt.title("Noisy colour-redshift relation")
plt.tight_layout()
plt.show()

## Truncated label support and Distribution Shift

In astronomy, spectroscopic labels are typically **magnitude-limited** and biased toward bright, low-redshift objects. 

We simulate an extreme form of spectroscopic incompleteness, producing truncated label support that mimics one consequence of magnitude-limited surveys: missing high-redshift labels.

Specifically, we restrict our training set to $z < 1.2$, while the full survey population spans $z \in [0, 2]$.

### Defining the "Shift"

Since models see **features** (colors), not **labels** (redshifts), the deployment mismatch combines several related phenomena:

* **Covariate Shift ($p_{\mathrm{test}}(x) \neq p_{\mathrm{train}}(x)$):**
  - The test galaxies have different colors than the training set, but some overlap remains. 
    - Predictions here are "familiar" but less certain.
* **Feature-Space Extrapolation ($x \notin \mathrm{supp}(p_{\mathrm{train}})$):**
  - The model encounters colors it has never seen. 
    - Here, predictions are unconstrained by data and are dictated entirely by the model's **inductive bias**.
* **Label-Space Truncation:** 
  - The model has no examples of $z > 1.2$. 
    - Because the color-redshift relation is noisy, some high-$z$ galaxies may still fall within the training "color box" (interpolation), while others will land outside it (extrapolation).

## Scientific Implications

Errors in photometric redshift estimation propagate into downstream scientific analyses through derived population-level quantities such as the redshift distribution, survey selection functions, and completeness corrections. 

In cosmological pipelines, these quantities may ultimately influence parameter inference (e.g. matter density or clustering amplitude), but the magnitude of this impact depends on how errors correlate with redshift, selection effects, and the structure of the inference procedure.

Thus, **structured, regime-dependent errors can be amplified when embedded in hierarchical inference pipelines** and the scientific risk arises less from average error and more from how error varies across observational regimes.

The validation set is drawn from the same truncated distribution as training, intentionally reproducing the standard IID evaluation protocol


In [ ]:
mask_train = z < 1.2

df_train = df[mask_train].copy()

X_labeled = df_train[["color"]]
y_labeled = df_train["z"]
X_train, X_val, y_train, y_val = train_test_split(
    X_labeled,
    y_labeled,
    test_size=0.25,
    random_state=SEED)

# keep realistic deployment test set = whole survey
df_test = pd.DataFrame({
    "color": color_test,
    "z": z_test
})

X_test = df_test[["color"]]
y_test = df_test["z"]


print("Train size        :", len(X_train))
print("Val size          :", len(X_val))
print("Survey (Test) size:", len(X_test))
print("\nTrain z  min:", round(y_train.min(),3))
print("Val z    min:", round(y_val.min(),3))
print("Survey z min:", round(y_test.min(),3))
print("\nTrain z  max:", round(y_train.max(),3))
print("Val z    max:", round(y_val.max(),3))
print("Survey z max:", round(y_test.max(),3))

The following boundary-based definition is intentionally simplistic; in higher dimensions, support estimation typically requires density-aware methods.

In this 1D toy example we approximate support using the observed feature interval rather than density estimation.


In [ ]:
train_color_min = X_train["color"].min()
train_color_max = X_train["color"].max()

mask_test_ifs = (
    (X_test["color"] >= train_color_min) &
    (X_test["color"] <= train_color_max)
)

mask_test_ofs = ~mask_test_ifs


In [ ]:
train_min = X_train["color"].min()
train_max = X_train["color"].max()
ood_mask = ((X_test["color"] < train_min) | (X_test["color"] > train_max))
print("Fraction Inside  Observed Feature Range:", round(mask_test_ifs.mean(), 3))
print("Fraction Outside Observed Feature Range:", round(mask_test_ofs.mean(), 3))
plt.hist(X_test["color"], bins=40, alpha=0.6, label="Survey")
plt.axvline(train_min, ls="--", lw=2, label="Train min")
plt.axvline(train_max, ls="--", lw=2, label="Train max")
plt.xlabel("Observed colour")
plt.ylabel("Count")
plt.title("Out-of-domain feature detection")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(y_test,  bins=30, alpha=0.5, label="Full survey")
plt.hist(y_train, bins=30, alpha=0.7, label="Labeled training sample")
plt.hist(y_val,   bins=30, alpha=0.7, label="Labeled validation sample")
plt.xlabel("Redshift")
plt.ylabel("Count")
plt.title("Training sample is not representative")
plt.legend()
plt.tight_layout()
plt.show()

## Models, Inductive Bias, and Extrapolation

Different ML models encode different assumptions about how the universe behaves outside the observed data.

We compare four regressors:
* Dummy baseline
* Linear Regression
* Random Forest
* Gradient Boosting

| Model                 | Inductive Bias                       | Extrapolation Behavior                         |
| --------------------- | ------------------------------------ | -----------------------------------------------|
| **Linear Regression** | Global affine trend                  | Extends global relation beyond observed support|
| **Random Forest**     | Piecewise local averaging            | Saturates near the edge of training support    |
| **Gradient Boosting** | Sequential local residual correction | Strong interpolation but limited extrapolation |
| **Dummy Regressor**   | Constant mean prediction             | No learned structure                           |

Inside the training distribution, flexible nonlinear models often perform well because they can approximate local structure efficiently.

Outside the training distribution, predictions become dominated by inductive bias rather than observed evidence:
* **Linear models** extrapolate by extending global trends.
* **Tree ensembles** cannot extrapolate smooth functional behavior, instead produce locally averaged predictions near the boundary of training data.

For standard regression trees with averaging leaves, predictions remain bounded by target values observed within nearby regions of feature space (**saturation**).

This distinction is especially important in astronomy, where scientific analyses frequently probe sparsely labeled or entirely unlabeled regimes.
> Under distribution shift, model selection becomes a comparison of assumptions about the unseen universe.

The central trade-off is therefore:
* **Interpolation:** flexible nonlinear models reduce variance and fit complex local structure effectively.
* **Extrapolation:** predictions are determined primarily by the structural assumptions of the algorithm rather than by data.
> Choosing a model implicitly chooses how extrapolation will fail.

The linear model appears more robust here partly because the toy data-generating process is smooth and monotonic; under nonlinear physical relations, global extrapolation can fail catastrophically.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

lin = LinearRegression()

gbr = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    random_state=SEED
)

dummy = DummyRegressor(strategy="mean")

models = {
    "Dummy Mean": dummy,
    "Linear": lin,
    "Random Forest": rf,
    "Gradient Boosting": gbr
}

predictions_val = {}
predictions_test = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions_val[name] = model.predict(X_val)
    predictions_test[name] = model.predict(X_test)

pred_lin_val  = predictions_val["Linear"]
pred_lin_test = predictions_test["Linear"]
pred_gbr_val  = predictions_val["Gradient Boosting"]
pred_gbr_test = predictions_test["Gradient Boosting"]
pred_rf_val  = predictions_val["Random Forest"]
pred_rf_test = predictions_test["Random Forest"]
print("Models trained:", list(models.keys()))

### Prediction vs. Color: Visualizing the Extrapolation

To understand model failure, we must shift from **redshift space ($z$)** to **feature space (color)**. This is crucial because:

> At prediction time, models only see color and must infer redshift from learned structure.

#### 1. Training Support Boundary

The vertical red lines mark the **range of colors seen during training**. Outside these boundaries, the model is in a state of **outside the observed feature range**.

#### 2. Inside Support (Interpolation)

Within the red lines, all models behave reasonably. Differences in this regime are driven by model **flexibility** (trees fitting the local scatter) versus **bias** (the linear model enforcing a straight line).

#### 3. Outside observed feature range (proxy for feature-space extrapolation)

Here we approximate support using the observed color range for interpretability, though true support is density-based.

In the "unseen" color regime, the **inductive bias** of each algorithm takes over:

* **Ensemble Trees (RF/GBR):** The curves flatten into a **constant prediction**. Because trees use leaf-averaging, they simply predict the mean of the nearest training "box." They cannot "imagine" a trend continuing upward.
* **Linear Regression:** The model continues its global trend indefinitely. While this looks more "natural," it may be physically incorrect if the true relationship changes slope at high redshift.

For supervised models, extrapolation is primarily determined by whether the input features lie outside the observed training support.

* A high-redshift galaxy with a "common" color will be **interpolated** (and likely underestimated).
* A low-redshift galaxy with an "unusual" color will trigger **extrapolation**.

Failure occurs when the feature space is poorly represented, regardless of what the true redshift is.

In [ ]:
# Sort by color for clean visualization
idx_color = np.argsort(X_test["color"].values)
color_sorted = X_test["color"].values[idx_color]
pred_rf_sorted  = pred_rf_test[idx_color]
pred_lin_sorted = pred_lin_test[idx_color]
pred_gbr_sorted = pred_gbr_test[idx_color]
z_sorted = y_test.values[idx_color]
# Training support in feature space
color_train_min = X_train["color"].min()
color_train_max = X_train["color"].max()
plt.figure(figsize=(9,6))

# Because the mapping is noisy, there is no single deterministic relation between color and redshift. 
# We therefore visualize the sampled population trend rather than a unique function.
plt.scatter(
    color_sorted,
    z_sorted,
    s=6,
    alpha=0.4,
    color="tab:green",
    label="Survey population")
# Model predictions
plt.plot(color_sorted, pred_rf_sorted,  label="Random Forest", alpha=0.8, color="tab:blue")
plt.plot(color_sorted, pred_lin_sorted, label="Linear", alpha=0.8, color="tab:orange")
plt.plot(color_sorted, pred_gbr_sorted, label="Gradient Boosting", alpha=0.8, color="tab:red")
# Mark training support in color
plt.axvline(color_train_min, color="black", ls="--", lw=2, label="Train color min/max")
plt.axvline(color_train_max, color="black", ls="--", lw=2)
plt.xlabel("Observed colour (feature space)")
plt.ylabel("Predicted redshift")
plt.title("Prediction as a function of observed colour (feature-space view)")
plt.legend()
plt.tight_layout()
plt.show()

## Why Global Metrics Can Be Misleading

The central danger in scientific ML is not necessarily low accuracy, but rather that **standard validation metrics appear correct while hiding catastrophic failures.**

A model can perform exceptionally well in-domain yet fail systematically out-of-domain, while still appearing competitive under aggregate metrics like:

$\mathrm{RMSE} = \sqrt{\mathbb{E}[(\hat{z}-z)^2]}$

Standard evaluation pipelines primarily validate **interpolation** rather than scientific generalization. These failures are hard to detect because:

* **Validation Loss:** Remains low and behaves normally within the training regime.
* **Residuals:** Look well-behaved and centered in-distribution.
* **Calibration:** Uncertainty estimates appear reliable within the dense training support.

Global metrics are fundamentally **dominated by high-density regions of the evaluation distribution**. 

By averaging performance across the whole dataset, they:

1. **Mix Regimes:** They blend results from the high-confidence interpolation regime with the low-confidence extrapolation regime.
2. **Hide Structured Failures:** They wash out systematic biases—such as redshift saturation—that only emerge at the boundaries of the data.

The breakdown only becomes visible when explicitly testing for **distribution shift**, such as evaluating high-redshift regimes or full survey populations where labeled data are sparse.

> If your evaluation only measures performance on data drawn from the same regime as your training set, you are not validating the model's physics—you are merely measuring its ability to memorize a local neighborhood.

In [ ]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)
    r2   = r2_score(y_true, y_pred)
    return {
        "Model": name,
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "R2": r2
    }

In [ ]:
def styled_table(df_metrics):
    max_bias = df_metrics[['Bias']].abs().max().max() 
    # Create a custom map: Red (at -1) -> Green (at 0) -> Red (at +1)
    colors = ["#e31a1c", "#33a02c", "#e31a1c"] # Red, Green, Red
    rgr_cmap = LinearSegmentedColormap.from_list("RedGreenRed", colors)
    # Create the styled table
    styled_metrics = df_metrics.style
    styled_metrics = styled_metrics.background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE'])
    styled_metrics = styled_metrics.background_gradient(cmap='RdYlGn', subset=['R2'])
    styled_metrics = styled_metrics.background_gradient(
        cmap=rgr_cmap, 
        subset=['Bias'],
        vmin=-max_bias, 
        vmax=max_bias
    )
    styled_metrics = styled_metrics.format(
        lambda x: "{:.3e}".format(x) if (x < -100 or x > 100) else "{:.3f}".format(x))
    return styled_metrics

In [ ]:
results_val = []
for name, pred in predictions_val.items():
    results_val.append(evaluate_model(name+" on Validation Set", y_val, pred))

df_metrics_val = pd.DataFrame(results_val).set_index("Model")
display(styled_table(df_metrics_val))

In [ ]:
results_test = []
for name, pred in predictions_test.items():
    results_test.append(evaluate_model(name+" on Test Set", y_test, pred))

df_metrics_test = pd.DataFrame(results_test).set_index("Model")
display(styled_table(df_metrics_test))

In [ ]:
results_test_ifs = []
for name, pred in predictions_test.items():
    results_test_ifs.append(evaluate_model(name +" Inside Observed Feature Range on Test Set", 
                                          y_test[mask_test_ifs], pred[mask_test_ifs]))

df_metrics_test_ifs = pd.DataFrame(results_test_ifs).set_index("Model")
display(styled_table(df_metrics_test_ifs))

In [ ]:
results_test_ofs = []
for name, pred in predictions_test.items():
    results_test_ofs.append(evaluate_model(name + " Outside Observed Feature Range on Test Set", 
                                  y_test[mask_test_ofs], pred[mask_test_ofs]))

df_metrics_test_ofs = pd.DataFrame(results_test_ofs).set_index("Model")
display(styled_table(df_metrics_test_ofs))

## Regime-Based Error Structure

Model behavior does not fail all at once; it degrades as **feature-space support** (the density of labeled training data) thins out. 

We categorize this into three regimes:

### 1. Interpolation Regime

* **Context:** Data is dense and well-represented.
* **Behavior:** Predictions are strongly constrained by observed labels.
* **Limitation:** Performance is capped only by observational noise and model flexibility.

### 2. Weak-Support Regime (Covariate Shift)

* **Context:** Test data overlaps with the training range, but examples are sparse or unrepresentative (e.g., high-redshift galaxies that happen to share colors with low-redshift ones).
* **Behavior:** Predictions become sensitive to **inductive bias** rather than evidence.
* **Danger:** Performance begins to degrade here even before "true" extrapolation occurs.

### 3. Extrapolation Regime

* **Context:** Features fall entirely outside the training support.
* **Behavior:** The model is effectively "guessing" based on its internal architecture:
* **Trees (RF/GBR):** in this setup, the tree ensembles saturate near the boundary of the observed training support.
* **Linear Models:** Predictions continue the global trend, which may or may not be physically valid.

> The transition from interpolation to extrapolation is a continuous spectrum of diminishing support. 

Errors often occur in the Weak-Support regime, where the model produces a value but the underlying physical mapping is no longer constrained.

In [ ]:
plt.scatter(y_test, pred_gbr_test, s=8, alpha=0.35, label="Gradient Boosting", color="tab:red")
plt.scatter(y_test, pred_rf_test, s=8, alpha=0.35, label="Random Forest", color="tab:blue")
plt.scatter(y_test, pred_lin_test, s=8, alpha=0.20, label="Linear", color="tab:orange")
plt.axvline(1.2, color="tab:green", linestyle="--", lw=2, label="Training limit")
plt.plot([0,2],[0,2], "k--", lw=2)
plt.xlabel("True redshift")
plt.ylabel("Predicted redshift")
plt.legend()
plt.tight_layout()
plt.show()

## Residuals and Systematic Bias

Consider the residual: 
* $r = \hat{z} - z$
Ideal behaviour:
* zero mean
* no dependence on z

Observed behaviour:
* Tree models
    * strong negative bias at high $z$
    * residuals grow approximately linearly at high $z$ due to prediction saturation
* Linear model
    * smooth structured bias across range
> Systematic redshift bias can propagate nontrivially into downstream cosmological inference.

### Bias as a function of redshift

We measure systematic error as a function of the physical variable.
- Flat line at 0 → unbiased model
- Downward slope at high z → systematic underestimation
- Sharp deviation indicates extrapolation failure

In [ ]:
resid_gbr_test = pred_gbr_test - y_test
resid_rf_test = pred_rf_test - y_test
resid_lin_test = pred_lin_test - y_test
plt.scatter(y_test, resid_rf_test, s=8, alpha=0.5, label="Random Forest", color="tab:blue")
plt.scatter(y_test, resid_lin_test, s=8, alpha=0.3, label="Linear", color="tab:orange")
plt.scatter(y_test, resid_gbr_test, s=8, alpha=0.1, label="Gradient Boosting", color="tab:red")

plt.axhline(0, color="black", ls="--")
plt.xlabel("True redshift")
plt.ylabel("Residual (pred - true)")
plt.title("Bias as a function of redshift")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def binned_bias(y_true, y_pred, bins):
    idx = np.digitize(y_true, bins)
    vals = []
    for i in range(1, len(bins)):
        m = idx == i
        vals.append(np.mean(y_pred[m] - y_true[m]) if m.sum() > 0 else np.nan )
    return np.array(vals)

In [ ]:
bins = np.linspace(0, 2.0, 9)
bias_rf_test  = binned_bias(y_test.values, pred_rf_test, bins)
bias_lin_test = binned_bias(y_test.values, pred_lin_test, bins)
bias_gbr_test = binned_bias(y_test.values, pred_gbr_test, bins)
centers = 0.5 * (bins[:-1] + bins[1:])

plt.plot(centers, bias_rf_test,  "o-", lw=2, label="Random Forest", color="tab:blue")
plt.plot(centers, bias_lin_test, "o-", lw=2, label="Linear", color="tab:orange")
plt.plot(centers, bias_gbr_test, "o-", lw=2, label="Grad Boost", color="tab:red")
plt.axhline(0, color="black", ls="--")
plt.xlabel("Redshift bin center")
plt.ylabel("Mean bias (pred - true)")
plt.title("Systematic bias vs redshift")
plt.legend()
plt.tight_layout()
plt.show()

## Uncertainty and Calibration

We define:

- Aleatoric uncertainty: irreducible ambiguity/noise in observables
- Epistemic uncertainty: uncertainty from limited knowledge or missing support

For Random Forest, consider the ensemble spread:

$\sigma(x) = \mathrm{std}(\text{tree predictions})$

as a heuristic measure of model disagreement. 

Important limitations:
- this is not a Bayesian posterior uncertainty
- it does not separate epistemic and aleatoric components
- it reflects agreement among models trained on the same data support

### Apparent overconfidence under extrapolation

In this toy example, ensemble spread remains comparatively small while error increases.

Under distribution shift, ensemble-based uncertainty estimates may become less informative about true predictive error. 

In regions with weak or no training support, ensemble disagreement can decrease even while absolute error increases, particularly when all models share similar structural limitations (e.g. tree partition averaging).

This produces a partial decoupling between predicted uncertainty and realized error:
- **within-distribution**: uncertainty and error are often correlated  
- **out-of-distribution**: this relationship can break down  

The key failure is not simply miscalibration, but a **loss of responsiveness of uncertainty to changes in data regime**.

Mathematically, we observe:

$$\sigma_{\mathrm{ensemble}}(x) \downarrow \quad \text{while} \quad |\hat{z}-z| \uparrow$$

in extrapolation regimes.

In [ ]:
# We use the standard deviation of the individual trees as our uncertainty proxy
all_tree_preds = np.stack([tree.predict(X_test.values) for tree in rf.estimators_], axis=0)
mean_pred_rf = all_tree_preds.mean(axis=0)
std_pred_rf = all_tree_preds.std(axis=0)

In [ ]:
idx = np.argsort(y_test.values)
plt.plot(mean_pred_rf[idx], label="RF mean prediction", lw=1, alpha=0.6,color="tab:orange")
plt.plot(y_test.values[idx], label="True z", lw=1,color="tab:red")
plt.fill_between(
    np.arange(len(idx)),
    mean_pred_rf[idx] - std_pred_rf[idx],
    mean_pred_rf[idx] + std_pred_rf[idx],
    alpha=0.9,
    label=r"$\pm 1 \sigma$ ensemble spread",color="tab:blue")

plt.xlabel("Galaxies sorted by true redshift")
plt.ylabel("Redshift")
plt.title("Prediction with uncertainty proxy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
abs_err_rf = np.abs(pred_rf_test - y_test)
max_val_rf = max(std_pred_rf.max(), abs_err_rf.max())

plt.figure(figsize=(10, 6))
plt.scatter(std_pred_rf[mask_test_ifs], abs_err_rf[mask_test_ifs], color='tab:green', alpha=0.3, 
            label='Inside Observed Feature Range')
plt.scatter(std_pred_rf[mask_test_ofs], abs_err_rf[mask_test_ofs], color='tab:red', alpha=0.3, 
            label='Outside Observed Feature Range')
plt.plot([0, max_val_rf], [0, max_val_rf], 'k--', alpha=0.7, label='Perfect uncertainty–error correspondence')
plt.xlim(0, 0.15)
plt.ylim(0, 1.0)
plt.xlabel("Predicted Uncertainty (Tree $\sigma$)")
plt.ylabel("Realized Absolute Error ($|\hat{z} - z|$)")
plt.title("Overconfident uncertainty under extrapolation")
plt.legend(loc='upper right')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

If ensemble spread tracked predictive error, we would expect substantial overlap between realized outcomes and heuristic spread intervals.

Since RF $\sigma$ is not a probabilistic predictive standard deviation, interval coverage should be interpreted qualitatively rather than probabilistically.


In [ ]:
inside_rf = (
    (y_test[mask_test_ofs] >= mean_pred_rf[mask_test_ofs] - std_pred_rf[mask_test_ofs]) &
    (y_test[mask_test_ofs] <= mean_pred_rf[mask_test_ofs] + std_pred_rf[mask_test_ofs]))

coverage_rf = inside_rf.mean()
print("This is a heuristic sanity check, not a formal calibration test.")
print("Fraction of Outside Observed Feature Range objects inside ±1σ interval:", round(float(coverage_rf),4))
print("Mean σ OOFR:", std_pred_rf[mask_test_ofs].mean())
print("Mean absolute error OOFR:", abs_err_rf[mask_test_ofs].mean())

To determine if the model’s internal uncertainty ($\sigma$) is a reliable predictor of its actual performance, we perform the following steps:
- **Quantile Binning**: It divides the galaxies into 7 equal-sized groups (quantiles) based on their predicted uncertainty (std_pred). This ensures each bin has a statistically significant number of samples.
- **Averaging**: Within each bin, it calculates the Mean Predicted σ (what the model thinks the error is) vs. the Mean Absolute Error (what the error actually is).
- **Diagonal Comparison**: It plots these averages against a 1:1 identity line (the dashed black line).

Then we consider the model: 
- **Better-Calibrated**: If larger predicted uncertainty tends to correspond to larger realized errors, the uncertainty proxy is at least directionally informative.
- **Overconfident**: If realized errors grow substantially faster than predicted uncertainty, the model is underestimating its uncertainty or failing under distribution shift.
- **Underconfident**: If predicted uncertainty is systematically larger than realized errors, the model is overly conservative.

In [ ]:
# If predicted uncertainty is meaningful, larger predicted sigma should correspond to larger realized errors.
bins_rf = np.quantile(std_pred_rf, np.linspace(0,1,8))
bins_rf[0] -= 1e-8
bins_rf[-1] += 1e-8
digitized_rf= np.digitize(std_pred_rf, bins_rf)
x_sigma_rf = []
y_err_rf = []
for i in range(1, len(bins_rf)):
    m_rf = digitized_rf == i
    if m_rf.sum() > 10:
        x_sigma_rf.append(std_pred_rf[m_rf].mean())
        y_err_rf.append(abs_err_rf[m_rf].mean())

plt.plot(x_sigma_rf, y_err_rf, "o-", lw=2)
plt.plot([0, max_val_rf], [0, max_val_rf], 'k--', alpha=0.7, label='Perfect uncertainty–error correspondence')
plt.xlim(0, 0.08)
plt.ylim(0, 0.7)
plt.xlabel("Mean predicted σ")
plt.ylabel("Mean absolute error")
plt.title("Uncertainty–error consistency check")
plt.tight_layout()
plt.show()

## NGBoost Baseline: Probabilistic Gradient Boosting

We introduce **Natural Gradient Boosting (NGBoost)** as a probabilistic regression baseline.

Unlike standard regressors that produce only point predictions, NGBoost predicts a conditional probability distribution:

$p(z \mid x) = \mathcal{N}(\mu(x), \sigma(x))$

The Gaussian assumption is itself a modeling choice and may fail when predictive distributions are asymmetric or multimodal, as often occurs in photometric-redshift estimation.

This allows the model to output:
* a mean prediction,
* an input-dependent predictive uncertainty.

Importantly, this is a modeled predictive distribution, not necessarily the true posterior over redshift, and probabilistic predictions do not automatically solve distribution shift.

Even when predictive distributions are well optimized in-domain:
* uncertainty may still become miscalibrated outside training support,
* predictive variance is not guaranteed to represent epistemic uncertainty,
* extrapolation failures can remain highly systematic.

That is the learned predictive scale primarily reflects modeled conditional dispersion rather than a principled estimate of epistemic uncertainty.

> The key scientific question is therefore whether uncertainty remains reliable when the model leaves the observed training regime.

In [ ]:
ngb = NGBRegressor(
    Dist=Normal, 
    n_estimators=10000, 
    learning_rate=0.001,      
    Base=DecisionTreeRegressor(
        max_depth=4, 
        min_samples_leaf=15,   
        max_features='sqrt'   
    ),
    col_sample=0.7,
    verbose_eval=100           
)
ngb.fit(
    X_train, y_train, 
    X_val=X_val, Y_val=y_val, 
    early_stopping_rounds=400 
)
preds_ngb_test = ngb.pred_dist(X_test)
mean_pred_ngb = preds_ngb_test.loc
std_pred_ngb = preds_ngb_test.scale

In [ ]:
idx = np.argsort(y_test.values)
plt.plot(mean_pred_ngb[idx], label="NGB mean prediction", lw=1, alpha=0.6,color="tab:orange")
plt.plot(y_test.values[idx], label="True z", lw=1,color="tab:red")
plt.fill_between(
    np.arange(len(idx)),
    mean_pred_ngb[idx] - std_pred_ngb[idx],
    mean_pred_ngb[idx] + std_pred_ngb[idx],
    alpha=0.9,
    label=r"$NGB \pm 1 \sigma$ ensemble spread",color="tab:blue")

plt.axvline(np.sum(y_test.values[idx] < 1.2), color="red", ls="--", label="Train boundary")
plt.xlabel("Sorted galaxies")
plt.ylabel("Redshift")
plt.title("NGBoost uncertainty behavior")
plt.legend()
plt.tight_layout()
plt.show()

## NGBoost vs. Random Forest

The goal of this comparison is not to determine which model is universally “better,” but to examine how different approaches represent predictive uncertainty under distribution shift.

###### Random Forest Uncertainty

For Random Forests, we used the spread between tree predictions: $\sigma_{\mathrm{RF}}(x)=\mathrm{std}(\text{tree predictions})$ as a practical disagreement measure.

Important limitations:
* this is not a Bayesian posterior uncertainty
* ensemble disagreement is not equivalent to epistemic uncertainty
* all trees inherit the same training-support limitations

Under extrapolation, tree predictions often saturate near the edge of the observed label range. 

Because many trees fail similarly, ensemble disagreement may remain artificially small even while prediction error grows substantially. This produces:
> overconfident failure under domain shift

###### NGBoost Predictive Uncertainty

NGBoost predicts the parameters of a conditional probability distribution: $p(z \mid x)=\mathcal{N}(\mu(x), \sigma(x))$

Unlike standard regressors that output only point estimates, NGBoost explicitly models an input-dependent predictive scale.

This predictive variance may reflect:
* observational noise
* unresolved structure in the data
* increased ambiguity in difficult regions of feature space

However, probabilistic predictions are not automatically calibrated under distribution shift. In particular:
* predictive uncertainty does not necessarily correspond to epistemic uncertainty
* uncertainty quality must still be evaluated empirically
* good likelihood optimization does not guarantee reliable OOD behavior

| Model         | Uncertainty Behavior                         |
| ------------- | -------------------------------------------- |
| Random Forest | disagreement-based heuristic uncertainty     |
| NGBoost       | explicit conditional predictive distribution |

The central scientific question is:
- not whether uncertainty exists, 
- but whether the model correctly signals reduced reliability in regions where the training data no longer constrain the inference problem.

In [ ]:
abs_err_ngb = np.abs(mean_pred_ngb - y_test.values)
max_val_ngb = max(std_pred_ngb.max(), abs_err_ngb.max())

plt.figure(figsize=(8, 6))
plt.scatter(std_pred_rf[mask_test_ifs], abs_err_rf[mask_test_ifs], color='tab:green', alpha=0.3, 
            label='Inside Observed Feature Range')
plt.scatter(std_pred_rf[mask_test_ofs], abs_err_rf[mask_test_ofs], color='tab:red', alpha=0.3, label='Outside Observed Feature Range')
plt.scatter(std_pred_ngb[mask_test_ifs],  abs_err_ngb[mask_test_ifs],  color='tab:olive', alpha=0.5, 
            label='NGB Inside Observed Feature Range')
plt.scatter(std_pred_ngb[mask_test_ofs], abs_err_ngb[mask_test_ofs], color='tab:pink', alpha=0.5, label='NGB Outside Observed Feature Range')
plt.plot([0, max_val_ngb], [0, max_val_ngb], 'k--', alpha=0.7, label='Perfect uncertainty–error correspondence')
plt.xlim(0, 0.175)
plt.ylim(0, 1.0)
plt.xlabel("Predicted Uncertainty (Tree $\sigma$)")
plt.ylabel("Realized Absolute Error ($|\hat{z} - z|$)")
plt.title("Overconfident uncertainty under extrapolation")
plt.legend(loc='upper right')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
bins_ngb = np.quantile(std_pred_ngb, np.linspace(0,1,8))
bins_ngb[0] -= 1e-8
bins_ngb[-1] += 1e-8
digitized_ngb = np.digitize(std_pred_ngb, bins_ngb)
x_sigma_ngb = []
y_err_ngb = []
for i in range(1, len(bins_ngb)):
    m_ngb = digitized_ngb == i
    if m_ngb.sum() > 10:
        x_sigma_ngb.append(std_pred_ngb[m_ngb].mean())
        y_err_ngb.append(abs_err_ngb[m_ngb].mean())

plt.plot(x_sigma_rf, y_err_rf, "o-", lw=2, label="RF")
plt.plot(x_sigma_ngb, y_err_ngb, "o-", lw=2, label="NGB")
plt.plot([0, max_val_ngb], [0, max_val_ngb], 'k--', alpha=0.7, label='Perfect uncertainty–error correspondence')
plt.xlim(0, 0.175)
plt.ylim(0, 0.6)
plt.legend()
plt.xlabel("Mean predicted σ")
plt.ylabel("Mean absolute error")
plt.title("Uncertainty calibration check")
plt.tight_layout()
plt.show()

In [ ]:
inside_ngb = (
    (y_test[mask_test_ofs] >= mean_pred_ngb[mask_test_ofs] - std_pred_ngb[mask_test_ofs]) &
    (y_test[mask_test_ofs] <= mean_pred_ngb[mask_test_ofs] + std_pred_ngb[mask_test_ofs]))

coverage_ngb = inside_ngb.mean()
print("Fraction of Outside Observed Feature Range objects inside ±1σ interval:", round(float(coverage_ngb),4))
print("Mean σ OOFR:", std_pred_ngb[mask_test_ofs].mean())
print("Mean absolute error OOFR:", abs_err_ngb[mask_test_ofs].mean())

## Failure Modes in Astronomical ML

Not all errors in scientific ML are equally fundamental. Failures occur at three levels:

###### Level 1 — Structural (Most Fundamental)

Arise from the data-generating process:
- selection bias (non-representative training sample)  
- incomplete observational support (e.g. missing high-z labels)  
- hidden / latent variables (e.g. dust, environment)  

These determine which statistical relationships are recoverable from the available data.
> Severe Level 1 failures can fundamentally limit which statistical relationships are recoverable from the available data.

###### Level 2 — Representational (Observation Limits)

Arise from how the universe is measured:
- photometric compression of spectra  
- degeneracy in observables (many-to-one mappings)  
- limited or missing features  

These determine whether the inverse problem is identifiable.

###### Level 3 — Algorithmic (Model Choice)

Arise from the learning algorithm:
- linear vs nonlinear assumptions  
- tree ensembles vs parametric models  
- inductive bias differences  

These determine how well structure is fit *within observed support*.

Practical issues:
- **Selection bias** → training sample not representative of the universe  
- **Distribution shift** → test population extends beyond training support  
- **Extrapolation failure** → tree-based models saturate at training bounds  
- **Hidden variables** → missing physics induces degeneracy  

Most ML evaluation emphasizes Level 3 (models), while scientific failure is dominated by Levels 1–2 (data + physics).
> Algorithmic improvements alone cannot recover information that is absent from the observations or training distribution.

## Fundamental Principle of Scientific ML

A model can only generalize reliably when:
1. the observables contain sufficient information about the target variable,
2. the training distribution overlaps the deployment distribution,
3. the model assumptions remain approximately valid in the prediction regime.
Failure of any of these conditions can produce scientifically unreliable inference even when standard validation metrics appear strong.

In scientific machine learning, predictive accuracy alone is therefore insufficient.

Reliable scientific inference requires understanding:
* where the model is supported by data,
* where it is extrapolating,
* and whether its uncertainty remains trustworthy under distribution shift.

Machine learning models do not discover arbitrary physical truth from finite data.

They interpolate statistical structure within the support of the observations and extrapolate according to inductive bias outside it.

When the data do not constrain a region of parameter space, model behavior is determined more by assumptions than evidence.


## Final Synthesis: Performance as a Function of Support

The experiments demonstrate that predictive reliability is not a global constant, but a function of **feature-space support**.

### 1. The Regime Dependency

* **Interpolation (High Support):** Models are data-driven. They achieve high accuracy, unbiased residuals, and more reliable uncertainty estimates.
* **Extrapolation (Low Support):** Models are bias-driven. Predictions are governed by **inductive bias** (e.g., tree saturation vs. linear divergence) rather than evidence.

### 2. The Failure of Global Metrics

Aggregate scores (RMSE, $R^2$) are statistically "blind" to regime shifts because:

* **Density Bias:** They are dominated by the high-density training distribution.
* **Error Averaging:** They wash out structured, catastrophic failures at the boundaries by averaging over heterogeneous regimes.

### 3. Scientific Implications

A model can be **globally accurate yet scientifically invalid** if it fails systematically in specific regimes (high-$z$ or faint objects) required for the analysis.

> Robust scientific ML requires evaluation strategies that explicitly partition performance by **support regime** (interpolation vs. extrapolation) rather than relying on global summaries.